# Fine-tune Qwen 2.5 7B for Voron 0.2 diagnosis (free GPU)

**Kaggle setup (once):** Settings (right panel) → *Accelerator*: **GPU T4 x2** or **GPU P100**; *Internet*: **On** (needs a phone-verified Kaggle account).
**Data:** *Add Input* → *Upload* → a new dataset with `train.jsonl` and `val.jsonl` from `agent/data/finetune/` (name it `voron-finetune`).
Then **Run All**. Training takes about 1–3 hours; the GGUF export about 10–20 minutes. Download the files listed in the last cell.

Method: QLoRA (4-bit base model + small LoRA adapter, ~1 % extra weights) with Unsloth. Loss is computed only on the assistant's replies.
Nothing from the 58 test cases is in this data.

In [ ]:
%%capture
!pip install -q unsloth
# if Unsloth complains about versions, restart the session and run this cell again

In [ ]:
# ---- settings for this run ----
DATA_VERSION = 'v2'          # 'v1' = train.jsonl / val.jsonl,  'v2' = train_v2.jsonl / val_v2.jsonl (with the simulation)
MODEL_NAME = 'voron-qwen-v2' # name of the model file and of the Hugging Face repository
EPOCHS = 2                   # passes over the training data

In [ ]:
import json, os, random
want = '_v2' if DATA_VERSION == 'v2' else ''
found = {}
for base in ('/kaggle/input', '/content'):
    for root, dirs, files in os.walk(base):
        for f in files:
            name = f.lower()
            for kind in ('train', 'val'):
                if name == f'{kind}{want}.jsonl': found[kind] = os.path.join(root, f)
print('found:', found)
assert 'train' in found and 'val' in found, f'Upload train{want}.jsonl and val{want}.jsonl first (see the first cell)'
train = [json.loads(l) for l in open(found['train'])]
val = [json.loads(l) for l in open(found['val'])]
print(len(train), 'train examples,', len(val), 'validation examples')

In [ ]:
from unsloth import FastLanguageModel
MAX_LEN = 3072
model, tokenizer = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-7B-Instruct-bnb-4bit', max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')
def to_text(ex): return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}
ds_train = Dataset.from_list([{'messages': e['messages']} for e in train]).map(to_text)
ds_val = Dataset.from_list([{'messages': e['messages']} for e in val[:200]]).map(to_text)
lens = [len(tokenizer(t)['input_ids']) for t in ds_train['text'][:200]]
print('tokens per example: median', sorted(lens)[len(lens)//2], 'max', max(lens))

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, eval_dataset=ds_val,
    dataset_text_field='text', max_seq_length=MAX_LEN, packing=False,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=EPOCHS,
        learning_rate=2e-4, warmup_steps=10, lr_scheduler_type='linear', weight_decay=0.01,
        logging_steps=10, eval_strategy='steps', eval_steps=100, save_strategy='no',
        optim='adamw_8bit', fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        seed=3407, output_dir='outputs', report_to='none'))
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train(); print(stats)

## Quick check on held-out faults (never seen in training)

In [ ]:
FastLanguageModel.for_inference(model)
def ask(msgs):
    p = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(p, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
random.seed(1); sample = random.sample(val, min(60, len(val)))
ok_json = ok_action = ok_cause = n_diag = ok_read = n_read = 0
for e in sample:
    gold = json.loads(e['messages'][2]['content']); txt = ask(e['messages'])
    try: pred = json.loads(txt[txt.find('{'):txt.rfind('}') + 1]); ok_json += 1
    except Exception: continue
    if e['meta']['role'] == 'reading':
        n_read += 1; ok_read += bool(pred.get('candidates')) and pred['candidates'][0].get('id') == gold['candidates'][0]['id']
    else:
        ok_action += pred.get('action') == gold['action']
        if gold['action'] == 'diagnose': n_diag += 1; ok_cause += pred.get('cause') == gold['cause']
print(f'valid JSON {ok_json}/{len(sample)} | reading: true part first {ok_read}/{n_read} | lead: same action {ok_action}/{len(sample) - n_read} | right cause when diagnosing {ok_cause}/{n_diag}')
print(ask(sample[0]['messages']))

## Save for Ollama (GGUF, 4-bit)

In [ ]:
model.save_pretrained_gguf(MODEL_NAME, tokenizer, quantization_method='q4_k_m')
import glob, shutil
src_files = glob.glob('/tmp/unsloth_saves/**/*.Q4_K_M.gguf', recursive=True) + glob.glob(f'{MODEL_NAME}*/*.Q4_K_M.gguf')
GGUF = f'/kaggle/working/{MODEL_NAME}.Q4_K_M.gguf'
shutil.copy(sorted(src_files, key=os.path.getmtime)[-1], GGUF); print('saved:', GGUF)

## Upload to Hugging Face (needs the Kaggle secret `HF_TOKEN`, a Write token)
Then on the laptop: `ollama pull hf.co/<your name>/<MODEL_NAME>-gguf` and `ollama cp hf.co/<your name>/<MODEL_NAME>-gguf <MODEL_NAME>`.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
user = api.whoami()['name']; repo = f'{user}/{MODEL_NAME}-gguf'
api.create_repo(repo, repo_type='model', exist_ok=True)
api.upload_file(path_or_fileobj=GGUF, path_in_repo=f'{MODEL_NAME}.Q4_K_M.gguf', repo_id=repo)
print('uploaded: https://huggingface.co/' + repo)
print(f'laptop:  ollama pull hf.co/{repo}   then   ollama cp hf.co/{repo} {MODEL_NAME}')